# Poisson: the learned operator

The solution operator $K = \Delta^{-1}: f \mapsto u$ of the 2-D Poisson problem $\Delta u = f$ on $(0,1)^2$, $u = 0$ on the boundary, is linear.

* Inputs and outputs are expanded in the sine basis $\xi_{\mathbf n}(x) = 2\sin(\pi n_1 x_1)\sin(\pi n_2 x_2)$, $\mathbf n \in [35]^2$ (1225 coefficients). $\Delta$ is diagonal in this basis, so $(Kf)_{\mathbf n} = -f_{\mathbf n} / (\pi^2 |\mathbf n|^2)$.
* Input measure: independent coefficients $f_{\mathbf n} \sim \mathrm{Jacobi}(\alpha_{\mathbf n}, \alpha_{\mathbf n})$ with $\alpha_{\mathbf n} = \|\mathbf n\|_1^3$. The 1225 modes carry 95% of the input energy.
* Approximation space: linear operators acting on the first $N_{\mathrm{eff}}$ input modes (lexicographic order), i.e. the index set $\{e_1, \dots, e_{N_{\mathrm{eff}}}\}$ (`owls.linear_index_set`).

Here we learn $\tilde K_V$ with $N_{\mathrm{eff}} = 1000$ from $M$ = `owls.sample_size(1000)` optimally sampled inputs. We plot the action of $\tilde K_V$ on the median-error test input, and the learned operator as a $1225 \times 1225$ matrix $\langle \tilde K_V \xi_{\mathbf n}, \xi_{\mathbf m}\rangle$, which should be diagonal. This runs in about a minute, so there is no saved data.

**Status:** rerun after the cleanup. In arXiv v1 the matrix plot showed the expansion coefficients, which differ from $\langle \tilde K_V \xi_{\mathbf n}, \xi_{\mathbf m}\rangle$ by a factor $1/\sigma_{\mathbf n}$ per row. See `ERRATA.md`.

In [ ]:
QUICK = False      # True: tiny version of the experiment, written to a temporary folder (for testing)

import os
import tempfile

import numpy as np
import matplotlib.pyplot as plt

import owls
from owls.plotting import paper_style, savefig
import matplotlib.colors as colors

OUT = tempfile.mkdtemp() if QUICK else "."
DATA, FIGS = os.path.join(OUT, "data"), os.path.join(OUT, "figures")
os.makedirs(DATA, exist_ok=True)
os.makedirs(FIGS, exist_ok=True)

In [ ]:
# Parameters
K_MODES = 35      # sine modes per direction (d = 35^2 = 1225)
N_EFF = 1000      # learn the action on the first 1000 input modes
M_TEST = 500
SEED = 0
if QUICK:
    K_MODES, N_EFF, M_TEST = 8, 40, 50

n1, n2 = np.meshgrid(np.arange(1, K_MODES + 1), np.arange(1, K_MODES + 1), indexing="ij")
n1, n2 = n1.flatten(), n2.flatten()                              # modes n = (n1, n2) in lexicographic order
measures = [owls.Jacobi((a + b) ** 3) for a, b in zip(n1, n2)]  # f_n ~ Jacobi(alpha_n, alpha_n), alpha_n = ||n||_1^3
eigs = -1 / (np.pi**2 * (n1**2 + n2**2))                        # K is diagonal: (Kf)_n = eigs_n * f_n
d = len(measures)                                                # number of input (and output) coefficients

# Fraction of the input energy E||f||^2 = sum_n 1/(2 alpha_n + 3) carried by these modes.
# (Over all n in N^2, group the terms by m = ||n||_1; there are m - 1 of them.)
m = np.arange(2, 2_000_000)
total = np.sum((m - 1) / (2.0 * m**3 + 3))
print(f"{d} input modes carry {sum(1 / (2 * ms.alpha + 3) for ms in measures) / total:.2%} of the input energy")

In [ ]:
rng = np.random.default_rng(SEED)
L = owls.linear_index_set(N_EFF, d)
model = owls.learn_operator(lambda f: f * eigs, measures, L, optimal=True, rng=rng)
print(f"M = {model.M}, cond(G) = {model.cond():.3f}")

x_test = owls.sample_rho(M_TEST, measures, rng)
y_test = x_test * eigs
y_pred = model.predict(x_test)
test_errors = np.linalg.norm(y_pred - y_test, axis=1)
print(f"Empirical Bochner error: {owls.bochner_error(y_pred, y_test):.3e}")

In [ ]:
fig_width, fig_height = paper_style(width=1.0, aspect=0.6, fontsize=9)

# Median-error test sample, evaluated on a grid
sample = np.argmin(np.abs(test_errors - np.median(test_errors)))
n_grid = 150
x = np.linspace(0, 1, n_grid, endpoint=False)
X, Y = np.meshgrid(x, x, indexing="ij")
V = np.stack([2 * np.sin(a * np.pi * X) * np.sin(b * np.pi * Y) for a, b in zip(n1, n2)], axis=-1).reshape(-1, d)

f_spatial = (V @ x_test[sample]).reshape(n_grid, n_grid)
u_true = (V @ y_test[sample]).reshape(n_grid, n_grid)
u_approx = (V @ y_pred[sample]).reshape(n_grid, n_grid)
abs_error = np.abs(u_true - u_approx)
print(f"Median test error: {test_errors[sample]:.3e}")

fig, axes = plt.subplots(1, 4, figsize=(fig_width, 1.3 * fig_height), layout="constrained")
im0 = axes[0].imshow(f_spatial, origin="lower", extent=[0, 1, 0, 1], cmap="coolwarm")
axes[0].set_title(r"$f\sim \rho$")
cbar0 = fig.colorbar(im0, ax=axes[0], location="left", fraction=0.05, pad=0.02)

vmin, vmax = min(u_true.min(), u_approx.min()), max(u_true.max(), u_approx.max())
axes[1].imshow(u_true, origin="lower", extent=[0, 1, 0, 1], cmap="viridis", vmin=vmin, vmax=vmax)
axes[1].set_title(r"$Kf$")
im2 = axes[2].imshow(u_approx, origin="lower", extent=[0, 1, 0, 1], cmap="viridis", vmin=vmin, vmax=vmax)
axes[2].set_title(r"$\tilde{K}_Vf$")
im3 = axes[3].imshow(abs_error, origin="lower", extent=[0, 1, 0, 1], cmap="hot",
                     norm=colors.LogNorm(abs_error.min() + 1e-8, abs_error.max()))
axes[3].set_title(r"$\vert Kf - \tilde{K}_Vf\vert $")
cbar3 = fig.colorbar(im3, ax=axes[3], location="right", fraction=0.05, pad=0.02)
fig.colorbar(im2, ax=axes, location="bottom", fraction=0.025, pad=0.02)  # shared by the two solution plots
for ax in axes:
    ax.axis("off")
    ax.set_aspect("equal")

# Shrink the side colorbars to the height of the images
fig.canvas.draw()
fig.set_layout_engine("none")
axpos, cpos = axes[3].get_position(), cbar3.ax.get_position()
h = 0.9 * axpos.height
y0 = axpos.y0 + (axpos.height - h) / 2
cbar3.ax.set_position([cpos.x0, y0, cpos.width, h])
cbar0.ax.set_position([cbar0.ax.get_position().x0, y0, cpos.width, h])
savefig(os.path.join(FIGS, "median_err.pdf"))
plt.show()

# The learned operator as a matrix: rows n (input mode), columns m (output mode)
K_matrix = model.as_matrix().T   # K_matrix[n, m] = < K_V xi_n, xi_m >
fig, ax = plt.subplots(figsize=(fig_width, fig_height), layout="constrained")
im = ax.imshow(K_matrix, norm=colors.SymLogNorm(vmin=K_matrix.min(), vmax=K_matrix.max(), linthresh=1e-8), cmap="magma")
ax.set_xlabel(r"$\mathbf{m}$", fontsize=14)
ax.set_ylabel(r"$\mathbf{n}$", fontsize=14)
ax.set(xticks=[], yticks=[])
ax.set_title(r"$\langle \tilde{K}_{V}\xi_\mathbf{n}, \xi_\mathbf{m}\rangle_{\mathcal{Y}}$", fontsize=16)
fig.colorbar(im, location="bottom", pad=0.02, fraction=0.05)
savefig(os.path.join(FIGS, "approx_operator.pdf"))
plt.show()